# H11: Gauge Fraction vs Batch-Dead Hidden Fraction
## Notebook counterpart to `run_experiment.py`

This notebook is the notebook-side counterpart to:

- `experiments/Tier_1_Core_Mechanism_Tests/H11_GAUGE_VS_DEAD_NEURONS/run_experiment.py`

It deliberately **does not re-implement the experiment core**. Instead, it imports the script, runs the script's `run_experiment()` / `analyze_results()` API, and then presents the returned results with tables, figures, and calibrated interpretation.

## Toy-scope framing

This is a **toy probe** of whether the raw gradient's gauge fraction changes as hidden units become batch-dead under a bias sweep in a small synthetic ReLU MLP. It is intended to clarify what is and is not visible in this setup, not to settle the full theory of Muon.

## What this notebook measures, and what it does not

### Measured here
- A **6-layer width-32 ReLU MLP** with biases, trained on synthetic Gaussian regression data.
- A sweep over bias initializations `+2, +1, 0, -1, -2, -3, -5`.
- **Hidden-layer batch-dead fraction**: a hidden neuron is counted as dead if it never activates on the sampled batch `X` used in the run.
- **Gauge fraction** of the weight gradient using the same Stiefel normal-space decomposition as the script.
- **NaN gauge regimes** when the squared gradient Frobenius norm is too small or non-finite for a reliable decomposition.
- Final training loss and simple early-stop diagnostics.

### Not measured here
- Global deadness over all possible inputs.
- Symmetry-group size or a direct measure of "local symmetry survives / global symmetry breaks."
- Muon updates, Muon-vs-SGD comparisons, or Muon's empirical benefit.
- Formal inferential statistics. The H1-H4 checks are heuristic diagnostics, not hypothesis tests in the statistical-testing sense.

### Primary analysis choice
The **main dead-vs-gauge summaries use hidden layers only**. The last layer has no ReLU and is kept separate as a control instead of being mixed into the primary dead-vs-gauge summary.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(str(obj))

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [ ]:
REPO_REL_SCRIPT = Path(
    "experiments/Tier_1_Core_Mechanism_Tests/H11_GAUGE_VS_DEAD_NEURONS/run_experiment.py"
)


def find_repo_root(start=None):
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / REPO_REL_SCRIPT).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate repo root containing {REPO_REL_SCRIPT} starting from {start}"
    )


REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / REPO_REL_SCRIPT.parent
SCRIPT_PATH = REPO_ROOT / REPO_REL_SCRIPT

spec = importlib.util.spec_from_file_location("h11_run_experiment_module", SCRIPT_PATH)
h11 = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(h11)

print(f"Repo root:      {REPO_ROOT}")
print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Script path:    {SCRIPT_PATH}")
print("Notebook import mode: explicit file-path import via importlib (no __file__ dependency)")

## Reproducibility / identity / runtime

This notebook uses the script as the single computational source of truth.

**Script entrypoint**
```bash
python experiments/Tier_1_Core_Mechanism_Tests/H11_GAUGE_VS_DEAD_NEURONS/run_experiment.py
```

**Notebook identity**
- Same experiment family: `H11_GAUGE_VS_DEAD_NEURONS`
- Same default configuration unless explicitly overridden
- Same returned raw results and same analysis logic

The next cell runs the experiment once through the script API and records basic runtime information.

In [ ]:
start_time = time.perf_counter()
experiment_results = h11.run_experiment(verbose=False)
analysis = h11.analyze_results(experiment_results)
elapsed_seconds = time.perf_counter() - start_time

config = experiment_results["config"]
seeds = experiment_results["seeds"]
condition_df = pd.DataFrame(analysis["condition_summaries"])

print(f"Notebook execution runtime for run_experiment() + analyze_results(): {elapsed_seconds:.2f} seconds")
print(f"Python: {platform.python_version()} | NumPy: {np.__version__} | pandas: {pd.__version__}")
print(f"Seeds: {seeds}")

In [ ]:
repro_df = pd.DataFrame([
    {"item": "experiment_name", "value": experiment_results["experiment_name"]},
    {"item": "script_path", "value": experiment_results["script_path"]},
    {"item": "command", "value": "python experiments/Tier_1_Core_Mechanism_Tests/H11_GAUGE_VS_DEAD_NEURONS/run_experiment.py"},
    {"item": "runtime_seconds_this_notebook_run", "value": round(elapsed_seconds, 3)},
    {"item": "dim", "value": config["dim"]},
    {"item": "num_layers", "value": config["num_layers"]},
    {"item": "num_samples", "value": config["num_samples"]},
    {"item": "num_steps", "value": config["num_steps"]},
    {"item": "lr", "value": config["lr"]},
    {"item": "momentum", "value": config["momentum"]},
    {"item": "bias_inits", "value": config["bias_inits"]},
    {"item": "seeds", "value": seeds},
    {"item": "grad_norm_threshold", "value": f"{config['grad_norm_threshold']:.1e}"},
    {"item": "loss_blowup_threshold", "value": config["loss_blowup_threshold"]},
])

display(repro_df)

## Condition-level summary

The table below is the notebook's primary summary table. It is intentionally aligned with the refactored script:

- **Hidden dead %** = mean hidden-layer batch-dead fraction
- **Hidden gauge %** = mean hidden-layer gauge fraction over valid hidden-layer gradient decompositions only
- **Hidden NaNs** = number of hidden-layer gauge entries that were undefined or unreliable
- **L6 gauge %** = last-layer control value, reported separately
- **Early stops** = number of seeds in that bias condition that stopped before the nominal 100 updates due to `NaN` or blow-up loss

In [ ]:
summary_table = condition_df[[
    "bias_init",
    "mean_hidden_dead_percent",
    "mean_hidden_gauge_percent",
    "valid_hidden_gauge_count",
    "nan_hidden_gauge_count",
    "mean_last_layer_gauge_percent",
    "mean_loss",
    "std_loss",
    "mean_steps_completed",
    "num_stopped_early",
]].copy()
summary_table = summary_table.rename(columns={
    "mean_hidden_dead_percent": "hidden_dead_%",
    "mean_hidden_gauge_percent": "hidden_gauge_%",
    "valid_hidden_gauge_count": "valid_hidden_gauge_count",
    "nan_hidden_gauge_count": "nan_hidden_gauge_count",
    "mean_last_layer_gauge_percent": "L6_gauge_%",
    "mean_loss": "mean_loss",
    "std_loss": "std_loss",
    "mean_steps_completed": "mean_steps_completed",
    "num_stopped_early": "early_stops",
})
display(summary_table.round(3))

In [ ]:
condition_level = analysis["condition_level"]
valid_conditions = condition_level["valid_condition_count"]
total_conditions = condition_level["total_condition_count"]
status = condition_level["status"]
pearson_r = condition_level["pearson_r"]

interp = f"""
### Condition-level interpretation

- In this run, **{valid_conditions}/{total_conditions}** bias conditions had a finite hidden-layer gauge summary.
- The condition-level correlation is **r = {pearson_r:.4f}** when computed on the valid conditions.
- Status: **{status}**.
- Because fewer than three bias conditions are valid here, the condition-level dead-vs-gauge correlation should be treated as **descriptive only**, not as serious evidence on its own.
- The table already shows an important confound: several conditions fall into **NaN / zero-gradient / unstable regimes**, so the absence of a gauge number is itself part of the result.
"""
display(Markdown(interp))

## Figure 1 — Bias sweep vs mean hidden batch-dead fraction

This figure shows the hidden-layer batch-dead fraction under the bias sweep, with individual seed means visible. The mean line is over seeds within each bias condition.

In [ ]:
bias_order = [row["bias_init"] for row in analysis["condition_summaries"]]
x_positions = np.arange(len(bias_order))
seed_dead_rows = []
for row in analysis["condition_summaries"]:
    for seed_idx, value in enumerate(row["seed_mean_hidden_dead_percent"]):
        seed_dead_rows.append({
            "bias_init": row["bias_init"],
            "seed_idx": seed_idx,
            "hidden_dead_%": value,
        })
seed_dead_df = pd.DataFrame(seed_dead_rows)

fig, ax = plt.subplots(figsize=(9, 5))
for idx, bias in enumerate(bias_order):
    sub = seed_dead_df[seed_dead_df["bias_init"] == bias].reset_index(drop=True)
    jitter = np.linspace(-0.12, 0.12, len(sub)) if len(sub) > 1 else np.array([0.0])
    ax.scatter(
        np.full(len(sub), x_positions[idx]) + jitter,
        sub["hidden_dead_%"],
        color="tab:blue",
        alpha=0.75,
        s=45,
        zorder=3,
    )

means = [row["mean_hidden_dead_percent"] for row in analysis["condition_summaries"]]
stds = [np.std(row["seed_mean_hidden_dead_percent"]) for row in analysis["condition_summaries"]]
ax.errorbar(x_positions, means, yerr=stds, color="black", marker="o", linewidth=2, capsize=4, label="condition mean ± seed std")
ax.set_xticks(x_positions)
ax.set_xticklabels([f"{b:+.0f}" for b in bias_order])
ax.set_xlabel("bias initialization")
ax.set_ylabel("mean hidden batch-dead fraction (%)")
ax.set_title("Bias sweep vs hidden batch-dead fraction")
ax.legend(loc="upper left")
plt.show()

In [ ]:
max_dead_bias = max(analysis["condition_summaries"], key=lambda row: row["mean_hidden_dead_percent"])
min_dead_bias = min(analysis["condition_summaries"], key=lambda row: row["mean_hidden_dead_percent"])

display(Markdown(f"""
### Figure 1 interpretation

- The bias sweep does change hidden-layer batch-deadness, but the realized pattern after training is **not a clean monotone 0%→100% ladder**.
- In this run, the least-dead condition is **bias {min_dead_bias['bias_init']:+.0f}** at **{min_dead_bias['mean_hidden_dead_percent']:.1f}%** mean hidden deadness.
- The most-dead conditions reach **{max_dead_bias['mean_hidden_dead_percent']:.1f}%** mean hidden deadness.
- This is one reason to present the actual post-training measurements rather than relying on initialization intuition alone.
"""))

## Figure 2 — Bias sweep vs mean hidden gauge fraction, with valid-count visibility

This is the primary gauge figure. Individual seed means are shown when they exist. The black line shows the condition mean hidden-layer gauge fraction, and the bars on the secondary axis show how many hidden-layer gauge entries were valid in that bias condition.

In [ ]:
seed_gauge_rows = []
for row in analysis["condition_summaries"]:
    for seed_idx, value in enumerate(row["seed_mean_hidden_gauge_percent"]):
        seed_gauge_rows.append({
            "bias_init": row["bias_init"],
            "seed_idx": seed_idx,
            "hidden_gauge_%": value,
        })
seed_gauge_df = pd.DataFrame(seed_gauge_rows)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

valid_counts = [row["valid_hidden_gauge_count"] for row in analysis["condition_summaries"]]
ax2.bar(x_positions, valid_counts, color="lightgray", alpha=0.5, width=0.6, label="valid hidden-layer gauge entries")
ax2.set_ylabel("valid hidden-layer gauge entries (max 25)")
ax2.set_ylim(0, 26)

for idx, bias in enumerate(bias_order):
    sub = seed_gauge_df[(seed_gauge_df["bias_init"] == bias) & (~seed_gauge_df["hidden_gauge_%"].isna())].reset_index(drop=True)
    if len(sub) == 0:
        continue
    jitter = np.linspace(-0.12, 0.12, len(sub)) if len(sub) > 1 else np.array([0.0])
    ax1.scatter(
        np.full(len(sub), x_positions[idx]) + jitter,
        sub["hidden_gauge_%"],
        color="tab:orange",
        alpha=0.85,
        s=45,
        zorder=3,
    )

mean_hidden_gauge = np.array([row["mean_hidden_gauge_percent"] for row in analysis["condition_summaries"]], dtype=float)
std_hidden_gauge = []
for row in analysis["condition_summaries"]:
    vals = np.array(row["seed_mean_hidden_gauge_percent"], dtype=float)
    vals = vals[~np.isnan(vals)]
    std_hidden_gauge.append(np.std(vals) if len(vals) > 0 else np.nan)
std_hidden_gauge = np.array(std_hidden_gauge, dtype=float)

finite_mask = np.isfinite(mean_hidden_gauge)
ax1.errorbar(
    x_positions[finite_mask],
    mean_hidden_gauge[finite_mask],
    yerr=std_hidden_gauge[finite_mask],
    color="black",
    marker="o",
    linewidth=2,
    capsize=4,
    label="condition mean ± seed std",
)
ax1.set_xticks(x_positions)
ax1.set_xticklabels([f"{b:+.0f}" for b in bias_order])
ax1.set_xlabel("bias initialization")
ax1.set_ylabel("mean hidden gauge fraction (%)")
ax1.set_title("Bias sweep vs hidden gauge fraction, with valid-count visibility")
ax1.set_ylim(0, 100)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper right")
plt.show()

In [ ]:
finite_hidden_gauge_conditions = [
    row for row in analysis["condition_summaries"] if np.isfinite(row["mean_hidden_gauge_percent"])
]

display(Markdown(f"""
### Figure 2 interpretation

- Only **{len(finite_hidden_gauge_conditions)}/{len(analysis['condition_summaries'])}** bias conditions produced a finite hidden-layer gauge summary in this run.
- That means the main condition-level dead-vs-gauge trend is heavily constrained by **NaN / zero-gradient / unstable regimes**.
- The notebook therefore shows the **valid-count bars explicitly**, so visually stable-looking means are not mistaken for broad support across the entire sweep.
- Where hidden-layer gauge is measurable at all, it remains in a roughly 50% range in this run; however, the low number of valid conditions is a major limitation.
"""))

## Per-layer tables

The next tables expose where deadness and gauge are sitting by layer, averaged over seeds. This is useful because a condition can have a finite mean while still hiding severe layerwise heterogeneity.

In [ ]:
per_layer_dead_df = pd.DataFrame([
    {
        "bias_init": row["bias_init"],
        **{name: value for name, value in zip(experiment_results["hidden_layer_names"], row["layer_dead_percent"])},
        "hidden_mean": row["mean_hidden_dead_percent"],
    }
    for row in analysis["per_layer_dead_table"]
])

per_layer_gauge_df = pd.DataFrame([
    {
        "bias_init": row["bias_init"],
        **{name: value for name, value in zip(experiment_results["layer_names"], row["layer_gauge_percent"])},
        "hidden_mean": row["mean_hidden_gauge_percent"],
        "L6_control_mean": row["mean_last_layer_gauge_percent"],
    }
    for row in analysis["per_layer_gauge_table"]
])

print("Per-layer hidden dead fraction (%)")
display(per_layer_dead_df.round(3))
print("Per-layer gauge fraction (%)")
display(per_layer_gauge_df.round(3))

In [ ]:
display(Markdown("""
### Per-layer interpretation

- The hidden-layer analysis is the primary object because those are the layers with ReLU gating and a meaningful dead-neuron notion.
- The last layer is shown for completeness and control, but it is not mixed into the main dead-vs-gauge summary.
- When a condition has many `NaN` gauge entries, that is not a missing cosmetic detail; it indicates that gauge decomposition was unavailable or unreliable in that regime.
"""))

## Figure 3 — Pooled hidden-layer dead-vs-gauge scatter and binned summary

This is the fine-grained analysis over individual hidden-layer points that had valid gauge decompositions. It is more detailed than the condition-level view, but it still does **not** create independent samples in a formal statistical sense because points are clustered within runs and conditions.

In [ ]:
pooled_points_df = pd.DataFrame(analysis["pooled_hidden_summary"]["points"])
bin_df = pd.DataFrame(analysis["pooled_hidden_summary"]["bin_summary"])

fig, ax = plt.subplots(figsize=(9, 5.5))
if not pooled_points_df.empty:
    unique_biases = list(dict.fromkeys(pooled_points_df["bias_init"].tolist()))
    cmap = plt.get_cmap("tab10")
    color_map = {bias: cmap(i % 10) for i, bias in enumerate(unique_biases)}
    for bias in unique_biases:
        sub = pooled_points_df[pooled_points_df["bias_init"] == bias]
        ax.scatter(
            sub["dead_percent"],
            sub["gauge_percent"],
            color=color_map[bias],
            alpha=0.75,
            s=45,
            label=f"bias {bias:+.0f}",
        )

populated_bins = bin_df[bin_df["n_points"] > 0].copy()
if not populated_bins.empty:
    centers = [0.5 * (lo + hi) for lo, hi in populated_bins["dead_bin"]]
    ax.errorbar(
        centers,
        populated_bins["mean_gauge_percent"],
        yerr=populated_bins["std_gauge_percent"],
        color="black",
        linewidth=2,
        marker="o",
        capsize=4,
        label="bin mean ± std",
    )

ax.set_xlabel("hidden-layer batch-dead fraction (%)")
ax.set_ylabel("hidden-layer gauge fraction (%)")
ax.set_title("Pooled hidden-layer dead vs gauge, valid points only")
ax.set_xlim(-2, 102)
ax.set_ylim(0, 100)
ax.legend(loc="best", ncol=2)
plt.show()

display(bin_df[["dead_bin_label", "n_points", "mean_gauge_percent", "std_gauge_percent"]].round(3))

In [ ]:
pooled = analysis["pooled_hidden_summary"]
display(Markdown(f"""
### Figure 3 interpretation

- The pooled hidden-layer analysis contains **N = {pooled['n_points']}** valid hidden-layer points.
- The pooled Pearson correlation is **r = {pooled['pearson_r']:.4f}**.
- This view is useful because it shows what happens **within the subset of hidden layers where gauge is measurable at all**.
- However, it should still be interpreted cautiously: these points are not independent experimental units, and the pooled analysis excludes the NaN regimes rather than solving them.
"""))

## Figure 4 — Last-layer control view

The final layer has no ReLU after it, so it is a control rather than part of the primary dead-neuron story. The point of showing it separately is transparency: it reveals whether the output-layer gauge summary remains available or disappears together with upstream gradient flow.

In [ ]:
last_layer_rows = []
for row in analysis["condition_summaries"]:
    for seed_idx, value in enumerate(row["seed_last_layer_gauge_percent"]):
        last_layer_rows.append({
            "bias_init": row["bias_init"],
            "seed_idx": seed_idx,
            "L6_gauge_%": value,
        })
last_layer_df = pd.DataFrame(last_layer_rows)
last_layer_summary_df = pd.DataFrame(analysis["last_layer_control"])

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
ax2.bar(
    x_positions,
    last_layer_summary_df["valid_last_layer_count"],
    color="lightgray",
    alpha=0.5,
    width=0.6,
    label="valid L6 seeds",
)
ax2.set_ylabel("valid L6 seeds (max 5)")
ax2.set_ylim(0, 5.5)

for idx, bias in enumerate(bias_order):
    sub = last_layer_df[(last_layer_df["bias_init"] == bias) & (~last_layer_df["L6_gauge_%"].isna())].reset_index(drop=True)
    if len(sub) == 0:
        continue
    jitter = np.linspace(-0.12, 0.12, len(sub)) if len(sub) > 1 else np.array([0.0])
    ax1.scatter(
        np.full(len(sub), x_positions[idx]) + jitter,
        sub["L6_gauge_%"],
        color="tab:green",
        alpha=0.85,
        s=45,
        zorder=3,
    )

finite_l6 = last_layer_summary_df["mean_last_layer_gauge_percent"].to_numpy(dtype=float)
finite_mask = np.isfinite(finite_l6)
ax1.plot(x_positions[finite_mask], finite_l6[finite_mask], color="black", marker="o", linewidth=2, label="mean L6 gauge")
ax1.set_xticks(x_positions)
ax1.set_xticklabels([f"{b:+.0f}" for b in bias_order])
ax1.set_xlabel("bias initialization")
ax1.set_ylabel("L6 gauge fraction (%)")
ax1.set_title("Last-layer control view")
ax1.set_ylim(0, 100)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper right")
plt.show()

display(last_layer_summary_df.round(3))

In [ ]:
display(Markdown("""
### Figure 4 interpretation

- The last layer is an explicit control, not part of the hidden dead-vs-gauge average.
- In the current run, the last-layer gauge summary is available only in a subset of bias conditions.
- That means the control itself reflects the same broad reality as the hidden layers: some conditions do not merely have "lower gauge"; they enter regimes where the decomposition is unavailable.
- The correct takeaway is therefore transparency about control behavior, not a claim that the last layer must always sit exactly near 50%.
"""))

## Final-loss / instability view

Because some conditions produce extreme losses or early stopping, it is important to surface instability directly rather than hiding it behind the dead-vs-gauge summaries.

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

mean_losses = condition_df["mean_loss"].to_numpy(dtype=float)
early_stops = condition_df["num_stopped_early"].to_numpy(dtype=float)

ax1.plot(x_positions, mean_losses, color="tab:red", marker="o", linewidth=2, label="mean final loss")
ax1.set_yscale("log")
ax1.set_xlabel("bias initialization")
ax1.set_ylabel("mean final loss (log scale)")
ax1.set_xticks(x_positions)
ax1.set_xticklabels([f"{b:+.0f}" for b in bias_order])
ax1.set_title("Final loss and early-stop count by bias condition")

ax2.bar(x_positions, early_stops, color="lightgray", alpha=0.5, width=0.6, label="early stops")
ax2.set_ylabel("early stops (max 5)")
ax2.set_ylim(0, 5.5)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper right")
plt.show()

In [ ]:
worst_loss_row = condition_df.loc[condition_df["mean_loss"].idxmax()]

display(Markdown(f"""
### Instability interpretation

- The worst-loss condition in this run is **bias {worst_loss_row['bias_init']:+.0f}**.
- Its mean final loss is **{worst_loss_row['mean_loss']:.4e}**, with **{int(worst_loss_row['num_stopped_early'])} / {len(seeds)}** early stops.
- This matters because a high-deadness / measurable-gauge point can still be scientifically confounded by a numerically unstable training regime.
"""))

## H1-H4 diagnostic outcomes

These are the same structured H1-H4 outcomes produced by the script analysis. They are retained as diagnostics, but should be read with their scope limitations in mind.

In [ ]:
tests_df = pd.DataFrame([
    {
        "test": record["name"],
        "question": record["question"],
        "status": record["status"],
        "result": record["result"],
    }
    for record in analysis["tests"]["records"]
])

display(tests_df)
print(f"Tests passed: {analysis['tests']['total_pass']}/{analysis['tests']['total_tests']}")

In [ ]:
h2_record = next(record for record in analysis["tests"]["records"] if record["name"] == "H2")
h4_record = next(record for record in analysis["tests"]["records"] if record["name"] == "H4")

conclusion_md = f"""
## Calibrated conclusion

### What this run supports
- This notebook and the script are now **computationally aligned by construction**: the notebook imports the script and uses the script's returned results.
- In this run, the primary hidden-layer summaries show that gauge is measurable only in a **restricted subset of conditions**.
- The pooled hidden-layer view gives **r = {analysis['pooled_hidden_summary']['pearson_r']:.4f}** over **N = {analysis['pooled_hidden_summary']['n_points']}** valid hidden-layer points.
- The condition-level view has **{analysis['condition_level']['valid_condition_count']}/{analysis['condition_level']['total_condition_count']}** valid conditions and is therefore **{analysis['condition_level']['status']}**.

### What this run does not support
- It does **not** show that gauge is globally constant across the full bias sweep.
- It does **not** directly measure Muon benefit.
- It does **not** establish a direct symmetry-count argument, because the implemented observables are batch-dead fraction, gauge fraction, loss, and NaN/validity structure.

### Most important caveats
- Several bias conditions land in **NaN / zero-gradient / unreliable-decomposition regimes**.
- The `+2` condition is also notable for **instability / blow-up behavior**, which means high-deadness and gauge observations there should be read alongside the loss explosion and early-stop information.
- The H2 diagnostic is **{h2_record['status']}**, which is the right outcome when the number of valid conditions is too small for a serious constancy claim.
- H4 is **{h4_record['status']}**, but even that pooled result should be read as descriptive because layer/seed points are not independent samples.

### Bottom line
This notebook now presents the current H11 experiment honestly as a **small synthetic probe**. It is useful for seeing where hidden-layer gauge remains measurable and where the experiment collapses into NaN or instability regimes, but it should not be overread as a direct measurement of Muon benefit or as a definitive theory test.
"""

display(Markdown(conclusion_md))